# Grok-ml-gpu-smoke

Kaggle T4×2 GPU smoke test pushed via Kaggle CLI.
Validates dual-GPU availability and a tiny PyTorch train step.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import torch
import torch.nn as nn

RESULTS = {
    "ok": False,
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
    "devices": [],
    "train_loss": None,
    "error": None,
}

def main():
    print("=" * 60)
    print("Grok-ml-gpu-smoke | Kaggle T4x2 validation")
    print("=" * 60)
    print(f"torch={torch.__version__}")
    print(f"cuda_available={torch.cuda.is_available()}")
    print(f"device_count={RESULTS['device_count']}")

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available — expected NvidiaTeslaT4 (T4x2)")

    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        props = torch.cuda.get_device_properties(i)
        info = {
            "index": i,
            "name": name,
            "total_memory_gb": round(props.total_memory / (1024**3), 2),
            "major": props.major,
            "minor": props.minor,
        }
        RESULTS["devices"].append(info)
        print(f"GPU[{i}]: {name} | {info['total_memory_gb']} GB")

    # Prefer dual GPU (T4x2). Still succeed on single T4 if quota maps that way.
    device = torch.device("cuda:0")
    model = nn.Sequential(
        nn.Linear(512, 1024),
        nn.ReLU(),
        nn.Linear(1024, 10),
    ).to(device)

    if torch.cuda.device_count() >= 2:
        model = nn.DataParallel(model)
        print("Using DataParallel across", torch.cuda.device_count(), "GPUs")
    else:
        print("WARNING: only 1 GPU visible; continuing on single GPU")

    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    t0 = time.time()
    model.train()
    for step in range(20):
        x = torch.randn(256, 512, device=device)
        y = torch.randint(0, 10, (256,), device=device)
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        opt.step()
        if step % 5 == 0 or step == 19:
            print(f"step={step:02d} loss={loss.item():.4f}")
    elapsed = time.time() - t0
    RESULTS["train_loss"] = float(loss.item())
    RESULTS["elapsed_sec"] = round(elapsed, 3)
    RESULTS["ok"] = True

    # Prove tensors land on GPU
    with torch.no_grad():
        probe = torch.randn(4, 512, device=device)
        out = model(probe)
        assert out.is_cuda, "output not on CUDA"
        print("probe_out_shape=", tuple(out.shape), "device=", out.device)

    out_path = Path("/kaggle/working/smoke_results.json")
    out_path.write_text(json.dumps(RESULTS, indent=2))
    print("wrote", out_path)
    print("SUCCESS")

try:
    main()
except Exception as e:
    RESULTS["error"] = f"{type(e).__name__}: {e}"
    Path("/kaggle/working/smoke_results.json").write_text(json.dumps(RESULTS, indent=2))
    print("FAILED:", RESULTS["error"], file=sys.stderr)
    raise
